# Restore MAGFiLO to the new Google Drive

**Updated Kaggle authorization diagnostics:** The notebook now validates the API token with Kaggle, displays the *Kaggle username* (not the secret token), and checks access before downloading.

Use the Kaggle account already registered for **Solar Filament Segmentation Challenge 2026**, which can be different from the Google account holding the target Drive. Generate an API token in [Kaggle API settings](https://www.kaggle.com/settings/api) while signed into the competition account. If access is denied, open the [competition rules](https://www.kaggle.com/competitions/filament-segmentation-2026/rules) with that same Kaggle account.

Run **Runtime → Restart session and run all**. When asked, enter the token in the hidden prompt only. A previously saved Colab secret `KAGGLE_API_TOKEN` may be from another account; if invalid, the notebook prompts for a replacement. Do not share the token in chat or add it to your public GitHub repository. Existing matching dataset files in the new Drive are verified with checksums rather than overwritten.


In [ ]:
%pip -q install -U kagglehub
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
creds, _ = google.auth.default()
drive_api = build('drive','v3',credentials=creds,cache_discovery=False)
FOLDER_ID = "1aA5QZgdnJVJdiudUAJWh2yoQN65oNmka"
target = drive_api.files().get(fileId=FOLDER_ID,fields='id,name,mimeType').execute()
assert target['mimeType']=='application/vnd.google-apps.folder', 'The Drive destination is not a folder.'
print('Verified the new Drive folder:',target['name'],target['id'])


In [ ]:
# Steps 2–4 run in one cell to prevent NameError when cells are executed out of order.
if 'drive_api' not in globals() or 'FOLDER_ID' not in globals():
    raise RuntimeError('First run the authentication cell immediately above this cell (Runtime > Run all).')

import os
from pathlib import Path
from getpass import getpass
import kagglehub
from kagglehub.exceptions import UnauthenticatedError

RULES = "https://www.kaggle.com/competitions/filament-segmentation-2026/rules"
SETTINGS = "https://www.kaggle.com/settings/api"

def authenticate_kaggle(token):
    token = token.strip()
    if token.startswith("{") or token.lower().endswith(".json"):
        raise ValueError(
            "A kaggle.json file or JSON was provided, not a Kaggle API token. "
            "Use 'Generate New Token' on your Kaggle API settings page, "
            "then paste ONLY its token value."
        )
    if not token:
        raise ValueError("No API token provided.")
    # Kaggle's modern API tokens normally begin with KGAT_.
    os.environ["KAGGLE_API_TOKEN"] = token
    kagglehub.config.set_kaggle_api_token(token)
    try:
        identity = kagglehub.whoami(verbose=False)
    except Exception as exc:
        raise RuntimeError(
            "Kaggle could not validate this token. Check that the value is the "
            "API token (not a username, account password, or kaggle.json file), "
            "and generate it in the Kaggle account joined to the competition: " + SETTINGS
        ) from exc
    username = identity.get("username")
    if not username:
        raise RuntimeError("Kaggle returned no authenticated username.")
    print("Kaggle identity VERIFIED:", username)
    return username

token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get("KAGGLE_API_TOKEN") or "").strip()
    except Exception:
        token = ""

print("Google Drive identity is separate from the Kaggle competition account.")
if not token:
    print("Open the API settings in the Kaggle account registered for this competition:", SETTINGS)
    token = getpass("Paste that account's NEW API token (hidden): ")
try:
    username = authenticate_kaggle(token)
except Exception as initial_auth_error:
    print("Kaggle authentication was not verified. Error:", type(initial_auth_error).__name__)
    print("If a token was cached from another account, generate a new token from the")
    print("Kaggle account that joined this competition. The token will not be printed.")
    replacement = getpass("Try a fresh competition-account token (hidden), or Enter to stop: ").strip()
    if not replacement:
        raise RuntimeError("Authentication failed. No valid Kaggle token available.") from initial_auth_error
    username = authenticate_kaggle(replacement)
finally:
    token = None

work = Path("/content/magfilo_kaggle_download")
work.mkdir(exist_ok=True)
for attempt in range(2):
    try:
        downloaded = Path(kagglehub.competition_download(
            "filament-segmentation-2026", output_dir=str(work)
        ))
        break
    except Exception as exc:
        if "401" not in str(exc) and "403" not in str(exc) and not isinstance(exc, UnauthenticatedError):
            raise
        print("Kaggle returned access denied for authenticated user:", username)
        print("Verify this EXACT user joined the competition:", RULES)
        if attempt == 0:
            print("If you joined from another Kaggle account, generate that account's token.")
            replacement_token = getpass(
                "Competition-account token to retry once (hidden), or Enter to stop: "
            ).strip()
            if replacement_token:
                username = authenticate_kaggle(replacement_token)
                replacement_token = None
                continue
        raise RuntimeError(
            "401/403 from Kaggle: the authenticated user lacks competition access "
            "or the rules/verification are not complete. Open the competition rules "
            "in the Kaggle account printed above before retrying."
        ) from exc

if not downloaded.exists():
    raise FileNotFoundError(f'Kaggle download returned a nonexistent path: {downloaded}')
print('Downloaded:', downloaded)

import zipfile, shutil
def valid_zip(p):
    try:
        with zipfile.ZipFile(p) as z:
            names=[x.lower().replace('\\','/') for x in z.namelist()]
        return (any('train_images/' in n and n.endswith(('.jpg','.jpeg')) for n in names)
                and any('test_images/' in n and n.endswith(('.jpg','.jpeg')) for n in names)
                and any('annotations' in n and n.endswith('.json') for n in names))
    except (OSError,zipfile.BadZipFile):
        return False

candidates=[downloaded] if downloaded.is_file() else list(downloaded.rglob('*'))
source_zip=next((p for p in candidates if p.is_file() and p.suffix.lower()=='.zip' and valid_zip(p)),None)
archive=Path('/content/filament-segmentation-2026.zip')
if source_zip:
    if source_zip.resolve()!=archive.resolve(): shutil.copyfile(source_zip,archive)
else:
    files=[p for p in candidates if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.json')]
    if not files: raise RuntimeError('No images or JSON were downloaded.')
    base=downloaded if downloaded.is_dir() else downloaded.parent
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
        for p in sorted(files): z.write(p,p.relative_to(base).as_posix())
if not valid_zip(archive): raise RuntimeError('Train/test images or annotations missing; upload aborted.')
with zipfile.ZipFile(archive) as z:
    print('Verified',sum(n.endswith(('.jpg','.jpeg')) for n in z.namelist()),'images and',sum(n.endswith('.json') for n in z.namelist()),'JSON files.')

import hashlib,math,json
from googleapiclient.http import MediaFileUpload
PART_SIZE=180*1024*1024
BUFFER=8*1024*1024
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as src:
        for data in iter(lambda:src.read(BUFFER),b''): h.update(data)
    return h.hexdigest()
files_to_upload=[archive]
parts=[]
with archive.open('rb') as source:
    for idx in range(1,math.ceil(archive.stat().st_size/PART_SIZE)+1):
        path=Path('/content') / (archive.name+f'.part{idx:02d}')
        remaining=min(PART_SIZE,archive.stat().st_size-(idx-1)*PART_SIZE)
        with path.open('wb') as out:
            while remaining:
                chunk=source.read(min(BUFFER,remaining))
                if not chunk: raise IOError('Unexpected end of ZIP')
                out.write(chunk);remaining-=len(chunk)
        parts.append({'name':path.name,'bytes':path.stat().st_size,'sha256':sha256(path)})
        files_to_upload.append(path)
manifest={'archive':archive.name,'bytes':archive.stat().st_size,'sha256':sha256(archive),'parts':parts}
manifest_file=Path('/content/filament-segmentation-2026.parts.json')
manifest_file.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
files_to_upload.append(manifest_file)
for path in files_to_upload:
    safe=path.name.replace("'","\\'")
    existing=drive_api.files().list(q=f"'{FOLDER_ID}' in parents and name = '{safe}' and trashed = false",
        fields='files(id,name,size,md5Checksum)',page_size=100).execute().get('files',[])
    if existing:
        if len(existing)!=1:
            raise FileExistsError(f'Multiple files exist with the same name: {path.name}')
        def md5_file(f):
            h=hashlib.md5()
            with f.open('rb') as inp:
                for block in iter(lambda:inp.read(BUFFER),b''):
                    h.update(block)
            return h.hexdigest()
        if existing[0].get('md5Checksum') == md5_file(path):
            print('Already uploaded and verified:',path.name)
            continue
        raise FileExistsError(f'Different existing file; not overwriting: {path.name}')
    media=MediaFileUpload(str(path),mimetype='application/zip' if path==archive else 'application/octet-stream',
        chunksize=8*1024*1024,resumable=True)
    request=drive_api.files().create(body={'name':path.name,'parents':[FOLDER_ID]},media_body=media,
        fields='id,name,size,md5Checksum')
    result=None
    while result is None:
        status,result=request.next_chunk()
    print('Saved:',result['name'],result.get('size',''), 'bytes')
print('FINISHED: data archive, chunks and integrity manifest are now in the new Drive account.')
